# ML Trading Strategy — XGBoost Classifier on SPX

**Objectif** : prédire si le SPX sera en hausse dans `HORIZON` jours.  
**Signal** : 1 = long, 0 = flat.  
**Données** : SPX (`^GSPC`), VIX (`^VIX`), VVIX (`^VVIX`).

---

### Structure du notebook
1. Setup & configuration
2. Chargement des données
3. Feature engineering
4. Train / Validation / Test split
5. Pipeline & scaling
6. XGBoost baseline
7. Hyperparameter tuning
8. Évaluation du meilleur modèle
9. Backtest & métriques financières

## 1 — Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay,
)
from xgboost import XGBClassifier

from modules import data_loader, features, backtest

plt.rcParams["figure.dpi"] = 110
sns.set_theme(style="whitegrid")

# ── Global parameters ──────────────────────────────────────────────────────
START_DATE  = "2010-01-01"   # Data start
END_DATE    = None            # None = today
HORIZON     = 5               # Forward return horizon (trading days)
SCALER      = "robust"        # "standard" | "robust" | "minmax"
RANDOM_SEED = 42

# Train / val / test split ratios
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
# test = remaining 15%

print("Setup OK — Horizon:", HORIZON, "days | Scaler:", SCALER)

## 2 — Chargement des données

In [ ]:
print("Downloading data from Yahoo Finance...")
universe = data_loader.load_universe(start=START_DATE, end=END_DATE)

spx  = universe["SPX"]
vix  = universe["VIX"]
vvix = universe["VVIX"]

print(f"\nSPX  : {spx.index[0].date()} → {spx.index[-1].date()}  ({len(spx)} rows)")
print(f"VIX  : {vix.index[0].date()} → {vix.index[-1].date()}  ({len(vix)} rows)")
print(f"VVIX : {vvix.index[0].date()} → {vvix.index[-1].date()}  ({len(vvix)} rows)")
spx.tail(3)

In [ ]:
# Quick visualisation
fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)

spx["Close"].plot(ax=axes[0], color="steelblue", lw=1)
axes[0].set_title("SPX (S&P 500)")

vix["Close"].plot(ax=axes[1], color="darkorange", lw=1)
axes[1].set_title("VIX")

vvix["Close"].plot(ax=axes[2], color="purple", lw=1)
axes[2].set_title("VVIX")

plt.tight_layout()
plt.show()

## 3 — Feature Engineering

Le module `features.build_feature_matrix` calcule automatiquement :

| Famille | Features |
|---|---|
| **Trend** | SMA 5/10/20/50/200, EMA 12/26, pentes MA, distances MA |
| **Momentum** | RSI 14, MACD, ROC 5/10/21/63j, Stoch %K/%D, Williams %R, CCI |
| **Volatilité** | Bollinger %B / largeur, ATR, HVol 21/63j |
| **Volume** | OBV, Vol z-score, Vol ratio |
| **Cross-asset** | VIX niveau/change/z-score/régime, VVIX niveau/change, ratio VVIX/VIX |

In [ ]:
df = features.build_feature_matrix(spx, vix=vix, vvix=vvix, horizon=HORIZON)

FEATURE_COLS = features.get_feature_columns(df, horizon=HORIZON)
TARGET_COL   = f"forward_label_{HORIZON}"

print(f"Features : {len(FEATURE_COLS)}")
print(f"Target   : {TARGET_COL}")
print(f"Rows before NaN drop : {len(df)}")

df = df.dropna(subset=FEATURE_COLS + [TARGET_COL])
print(f"Rows after  NaN drop : {len(df)}")

# Class balance
balance = df[TARGET_COL].value_counts(normalize=True)
print(f"\nClass balance:\n  Up (1): {balance.get(1,0):.1%}  |  Flat/Down (0): {balance.get(0,0):.1%}")

In [ ]:
# Feature correlation heatmap (top 30 by std)
top_features = (
    df[FEATURE_COLS].std().sort_values(ascending=False).head(30).index.tolist()
)
corr = df[top_features].corr()

plt.figure(figsize=(14, 11))
sns.heatmap(corr, cmap="coolwarm", center=0, annot=False,
            linewidths=0.3, vmin=-1, vmax=1)
plt.title("Feature Correlation Matrix (top 30 by std)")
plt.tight_layout()
plt.show()

## 4 — Train / Validation / Test Split

**Important** : on respecte l'ordre temporel — pas de `shuffle`.  
Aucune donnée future ne doit contaminer le train.

In [ ]:
n = len(df)
n_train = int(n * TRAIN_RATIO)
n_val   = int(n * VAL_RATIO)
n_test  = n - n_train - n_val

df_train = df.iloc[:n_train]
df_val   = df.iloc[n_train : n_train + n_val]
df_test  = df.iloc[n_train + n_val :]

X_train = df_train[FEATURE_COLS]
y_train = df_train[TARGET_COL]

X_val   = df_val[FEATURE_COLS]
y_val   = df_val[TARGET_COL]

X_test  = df_test[FEATURE_COLS]
y_test  = df_test[TARGET_COL]

print(f"Train : {len(df_train):>5} rows  {df_train.index[0].date()} → {df_train.index[-1].date()}")
print(f"Val   : {len(df_val):>5} rows  {df_val.index[0].date()} → {df_val.index[-1].date()}")
print(f"Test  : {len(df_test):>5} rows  {df_test.index[0].date()} → {df_test.index[-1].date()}")

In [ ]:
# Visualise the split on the SPX price
fig, ax = plt.subplots(figsize=(14, 4))

ax.plot(df_train.index, df_train["Close"], color="steelblue",  lw=1, label="Train")
ax.plot(df_val.index,   df_val["Close"],   color="darkorange", lw=1, label="Validation")
ax.plot(df_test.index,  df_test["Close"],  color="green",      lw=1, label="Test")

ax.axvline(df_val.index[0],  color="darkorange", lw=1.2, ls="--", alpha=0.7)
ax.axvline(df_test.index[0], color="green",      lw=1.2, ls="--", alpha=0.7)

ax.set_title("SPX — Train / Validation / Test Split")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 5 — Pipeline de prétraitement

On encapsule **imputation + scaling** dans un `sklearn.Pipeline`.  
Le paramètre `SCALER` (défini en section 1) contrôle le scaler utilisé :
- `"standard"` → `StandardScaler` (mean=0, std=1)
- `"robust"` → `RobustScaler` (médiane/IQR — moins sensible aux outliers)
- `"minmax"` → `MinMaxScaler` (range [0, 1])

In [ ]:
SCALERS = {
    "standard": StandardScaler(),
    "robust":   RobustScaler(),
    "minmax":   MinMaxScaler(),
}

def build_pipeline(scaler_name: str = "robust", model=None) -> Pipeline:
    """Return an end-to-end sklearn Pipeline."""
    if model is None:
        model = XGBClassifier(
            n_estimators=200,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            use_label_encoder=False,
            eval_metric="logloss",
            random_state=RANDOM_SEED,
            n_jobs=-1,
        )
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler",  SCALERS[scaler_name]),
        ("model",   model),
    ])

baseline_pipe = build_pipeline(SCALER)
print(baseline_pipe)

## 6 — XGBoost Baseline

In [ ]:
baseline_pipe.fit(X_train, y_train)

def evaluate(pipe, X, y, label=""):
    """Print classification metrics for a fitted pipeline."""
    pred = pipe.predict(X)
    acc  = accuracy_score(y, pred)
    prec = precision_score(y, pred, zero_division=0)
    rec  = recall_score(y, pred, zero_division=0)
    f1   = f1_score(y, pred, zero_division=0)
    print(f"{'─'*45}")
    print(f" {label}")
    print(f"{'─'*45}")
    print(f"  Accuracy  : {acc:.4f}")
    print(f"  Precision : {prec:.4f}")
    print(f"  Recall    : {rec:.4f}")
    print(f"  F1 Score  : {f1:.4f}")
    print()
    return {"Accuracy": acc, "Precision": prec, "Recall": rec, "F1": f1}

train_metrics   = evaluate(baseline_pipe, X_train, y_train, "Baseline — Train")
val_metrics     = evaluate(baseline_pipe, X_val,   y_val,   "Baseline — Validation")

In [ ]:
# Confusion matrix on validation set
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_estimator(
    baseline_pipe, X_val, y_val,
    display_labels=["Flat/Down", "Up"],
    cmap="Blues", ax=ax
)
ax.set_title("Baseline — Confusion Matrix (Validation)")
plt.tight_layout()
plt.show()

## 7 — Hyperparameter Tuning

On utilise `RandomizedSearchCV` avec `TimeSeriesSplit` pour respecter l'ordre temporel.  
Modifier `N_ITER` et `CV_FOLDS` pour contrôler le budget de calcul.

In [ ]:
from scipy.stats import randint, uniform

N_ITER   = 60    # Nombre de combinaisons testées
CV_FOLDS = 5     # Folds temporels

param_distributions = {
    "model__n_estimators":       randint(100, 600),
    "model__max_depth":          randint(2, 8),
    "model__learning_rate":      uniform(0.01, 0.2),
    "model__subsample":          uniform(0.5, 0.5),          # [0.5, 1.0]
    "model__colsample_bytree":   uniform(0.4, 0.6),          # [0.4, 1.0]
    "model__min_child_weight":   randint(1, 10),
    "model__gamma":              uniform(0, 0.5),
    "model__reg_alpha":          uniform(0, 1),               # L1
    "model__reg_lambda":         uniform(0.5, 2),             # L2
    "model__scale_pos_weight":   uniform(0.8, 1.4),           # class imbalance
}

tscv = TimeSeriesSplit(n_splits=CV_FOLDS)

search_pipe = build_pipeline(SCALER)

# Combine train+val for tuning
X_tv = pd.concat([X_train, X_val])
y_tv = pd.concat([y_train, y_val])

search = RandomizedSearchCV(
    estimator=search_pipe,
    param_distributions=param_distributions,
    n_iter=N_ITER,
    scoring="f1",
    cv=tscv,
    verbose=1,
    random_state=RANDOM_SEED,
    n_jobs=-1,
    refit=True,
)

print(f"Starting search: {N_ITER} iterations × {CV_FOLDS} folds = {N_ITER * CV_FOLDS} fits")
search.fit(X_tv, y_tv)
print(f"\nBest CV F1 : {search.best_score_:.4f}")

In [ ]:
# Best hyperparameters
best_params = search.best_params_
print("Best hyperparameters:")
for k, v in sorted(best_params.items()):
    print(f"  {k:40s}: {v}")

In [ ]:
# Visualise tuning results
results_df = pd.DataFrame(search.cv_results_)
results_df = results_df.sort_values("mean_test_score", ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(
    range(len(results_df)),
    results_df["mean_test_score"].values,
    "o-", color="steelblue", ms=4, lw=1
)
axes[0].set_xlabel("Rank")
axes[0].set_ylabel("CV F1")
axes[0].set_title("CV F1 by Rank")
axes[0].grid(alpha=0.3)

axes[1].scatter(
    results_df["param_model__learning_rate"],
    results_df["param_model__max_depth"],
    c=results_df["mean_test_score"],
    cmap="RdYlGn", s=50, edgecolors="k", lw=0.3
)
axes[1].set_xlabel("Learning Rate")
axes[1].set_ylabel("Max Depth")
axes[1].set_title("F1 by LR × Depth")
plt.colorbar(axes[1].collections[0], ax=axes[1], label="CV F1")

plt.tight_layout()
plt.show()

## 8 — Évaluation du meilleur modèle

In [ ]:
best_model = search.best_estimator_

print("═" * 50)
print(" BEST MODEL — PERFORMANCE SUMMARY")
print("═" * 50)
train_m = evaluate(best_model, X_train, y_train, "Train Set")
val_m   = evaluate(best_model, X_val,   y_val,   "Validation Set")
test_m  = evaluate(best_model, X_test,  y_test,  "Test Set  ✓")

summary = pd.DataFrame(
    {"Train": train_m, "Validation": val_m, "Test": test_m}
).T
print("\nSummary table:")
display(summary.style.format("{:.4f}").background_gradient(cmap="RdYlGn", axis=0))

In [ ]:
# Classification report
y_pred_test = best_model.predict(X_test)
print(classification_report(y_test, y_pred_test, target_names=["Flat/Down", "Up"]))

In [ ]:
# Confusion matrix — test set
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_estimator(
    best_model, X_test, y_test,
    display_labels=["Flat/Down", "Up"],
    cmap="Blues", ax=ax
)
ax.set_title("Best Model — Confusion Matrix (Test Set)")
plt.tight_layout()
plt.show()

In [ ]:
# Feature importance
xgb_model = best_model.named_steps["model"]
importance = pd.Series(
    xgb_model.feature_importances_,
    index=FEATURE_COLS
).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 8))
importance.head(30).sort_values().plot.barh(ax=ax, color="steelblue", edgecolor="k", lw=0.4)
ax.set_title("XGBoost Feature Importance — Top 30")
ax.set_xlabel("Importance")
plt.tight_layout()
plt.show()

## 9 — Backtest & métriques financières

On applique la stratégie **long / flat** sur le SPX :  
- `signal = 1` → position longue le lendemain  
- `signal = 0` → pas de position  

Les signaux sont générés sur **tout le dataset** (train+val+test) après re-fit du meilleur modèle pour avoir une courbe complète.  
L'évaluation **out-of-sample** est la période **test** uniquement.

In [ ]:
# Generate signals on full dataset (using best model fitted on train+val)
X_all = df[FEATURE_COLS]
all_signals = pd.Series(
    best_model.predict(X_all),
    index=df.index,
    name="signal"
)

spx_prices = df["Close"]

bt = backtest.Backtest(
    prices=spx_prices,
    signals=all_signals,
    transaction_cost_bps=5,
)
bt.run()

print(f"Backtest period : {bt.results.index[0].date()} → {bt.results.index[-1].date()}")
print(f"Total positions : {all_signals.sum()} days long / {len(all_signals)} total")

In [ ]:
# ── Full period metrics ───────────────────────────────────────────────────
print("\n📊  FULL PERIOD METRICS")
metrics_full = bt.get_metrics()
display(metrics_full)

In [ ]:
# ── Out-of-sample (test) metrics only ────────────────────────────────────
test_prices  = df_test["Close"]
test_signals = pd.Series(
    best_model.predict(X_test),
    index=df_test.index,
    name="signal"
)

bt_oos = backtest.Backtest(
    prices=test_prices,
    signals=test_signals,
    transaction_cost_bps=5,
)
bt_oos.run()

print("\n📊  OUT-OF-SAMPLE (TEST SET) METRICS")
metrics_oos = bt_oos.get_metrics()
display(metrics_oos)

In [ ]:
# ── Dashboard plot ────────────────────────────────────────────────────────
bt.plot(figsize=(16, 16))

In [ ]:
# ── Monthly returns heatmap ───────────────────────────────────────────────
bt.plot_monthly_heatmap(figsize=(14, 6))

In [ ]:
# ── Monthly returns table (raw) ───────────────────────────────────────────
monthly_tbl = bt.monthly_returns()
display(
    monthly_tbl.style
        .format(lambda x: f"{x:.1%}" if not pd.isna(x) else "")
        .background_gradient(cmap="RdYlGn", vmin=-0.1, vmax=0.1, axis=None)
)

---

## Aller plus loin — idées d'extensions

| Idée | Comment |
|---|---|
| Changer l'horizon | Modifier `HORIZON = 10` en section 1 |
| Ajouter des tickers | `data_loader.DEFAULT_TICKERS` + colonnes dans `features.py` |
| Threshold tuning | Ajuster le seuil de décision `predict_proba` > threshold |
| Walk-forward backtest | Découper en fenêtres glissantes de train/test |
| Ensemble de modèles | Combiner XGBoost + LightGBM + RandomForest |
| Position sizing | Pondérer la position par la probabilité prédite |
| Stop-loss | Ajouter une règle de sortie dans le module backtest |